# Track Stitching & Re-ID — Cleaning Step 1

Fixes the three data-quality problems before analysis:

1. **Re-ID** — the tracker gives one real player several `display_track_id`s over the
   video (fragmentation). Here you **group those fragments into one canonical ID**
   (the jersey number) so every player has a single identity for the whole clip.
2. **Team classification** — you assign each canonical ID to a team / role while
   stitching, so it is consistent across all frames.
3. **Missing frames** — once IDs are stable, short detection gaps are linearly
   interpolated per canonical ID.

### Workflow
1. Set paths in **Configuration**, run all cells up to the gallery.
2. The **gallery** shows one row per raw track ID: 3 crop thumbnails (first / middle /
   last appearance), its active frame range, and its tracker team colour.
3. For each row, type the **canonical ID** (jersey number) and pick the **team/role**.
   Fragments of the same player → give them the **same canonical ID**.
   Leave canonical ID blank (or 0) to **drop** that track (false detection).
4. Press **Apply mapping**, review the collision report, then run **Interpolate gaps**
   and **Save**.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
VIDEO_PATH   = "/content/playbook/HILAL-HAZM_match_B_up8.mp4"
TRACKS_CSV   = "/content/playbook/output/per_frame_tracks.csv"
CLEAN_CSV    = "/content/playbook/output/per_frame_tracks_clean.csv"
IDMAP_CSV    = "/content/playbook/output/id_map.csv"

BALL_CLASS   = 0
GK_CLASS     = 1
PLAYER_CLASS = 2
REF_CLASS    = 3
PEOPLE       = (GK_CLASS, PLAYER_CLASS, REF_CLASS)

THUMB_H      = 90     # thumbnail height in px
PAD          = 0.35   # bbox padding fraction for crops
MAX_GAP      = 25     # max consecutive missing frames to interpolate per ID

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import cv2, os
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display as ipy_display
from io import BytesIO
from PIL import Image as PILImage, ImageDraw

print("Imports OK")

In [ ]:
# ── Load tracks & summarise each raw track ID ──────────────────────────────
df = pd.read_csv(TRACKS_CSV)
df["cx"] = (df["x1"] + df["x2"]) / 2
df["cy"] = (df["y1"] + df["y2"]) / 2

cap          = cv2.VideoCapture(VIDEO_PATH)
FPS          = cap.get(cv2.CAP_PROP_FPS) or 25.0
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VID_W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
VID_H        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

people = df[df["class_id"].isin(PEOPLE)].copy()

def _mode(s):
    m = s.mode()
    return int(m.iloc[0]) if len(m) else -1

summary = (people.groupby("display_track_id")
           .agg(cls=("class_id", _mode),
                team=("team_id", _mode),
                n=("frame", "size"),
                f0=("frame", "min"),
                f1=("frame", "max"),
                cx=("cx", "mean"),
                cy=("cy", "mean"))
           .reset_index()
           .sort_values(["cls", "f0"]))
summary["dur_s"] = (summary["f1"] - summary["f0"]) / FPS

print(f"Video : {TOTAL_FRAMES} frames @ {FPS:.2f} fps  ({VID_W}x{VID_H})")
print(f"Raw people track IDs : {len(summary)}  "
      f"(GK={ (summary.cls==GK_CLASS).sum() }, "
      f"PL={ (summary.cls==PLAYER_CLASS).sum() }, "
      f"REF={ (summary.cls==REF_CLASS).sum() })")
ipy_display(summary.head(60))

In [ ]:
# ── Build crop-thumbnail strips for every raw track ID ─────────────────────
# Pick first / middle / last frame of each track, read each needed frame once,
# crop the padded bbox, and assemble a labelled horizontal strip per track.

def _sample_frames(row):
    f0, f1 = int(row.f0), int(row.f1)
    fm = (f0 + f1) // 2
    return sorted(set([f0, fm, f1]))

# frame -> list of (display_track_id, slot_index, bbox)
need = {}
samples = {}
for _, r in summary.iterrows():
    fr = _sample_frames(r)
    samples[int(r.display_track_id)] = fr
    for slot, f in enumerate(fr):
        need.setdefault(f, []).append((int(r.display_track_id), slot, f))

# crops[tid][slot] = RGB ndarray
crops = {int(t): {} for t in summary.display_track_id}
look = people.set_index(["frame", "display_track_id"])

for f in sorted(need):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, bgr = cap.read()
    if not ok:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    H, W = rgb.shape[:2]
    for tid, slot, fr in need[f]:
        try:
            row = look.loc[(f, tid)]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
        except KeyError:
            continue
        x1, y1, x2, y2 = row.x1, row.y1, row.x2, row.y2
        bw, bh = x2 - x1, y2 - y1
        px, py = bw * PAD, bh * PAD
        cx1 = max(0, int(x1 - px)); cy1 = max(0, int(y1 - py))
        cx2 = min(W, int(x2 + px)); cy2 = min(H, int(y2 + py))
        if cx2 - cx1 < 4 or cy2 - cy1 < 4:
            continue
        crops[tid][slot] = (rgb[cy1:cy2, cx1:cx2], fr)

def strip_jpeg(tid) -> bytes:
    fr_list = samples[tid]
    imgs = []
    for slot in range(len(fr_list)):
        if slot in crops[tid]:
            arr, fr = crops[tid][slot]
            im = PILImage.fromarray(arr)
            w = max(1, int(im.width * THUMB_H / im.height))
            im = im.resize((w, THUMB_H))
            d = ImageDraw.Draw(im)
            d.rectangle([0, 0, w-1, 12], fill=(0, 0, 0))
            d.text((2, 1), f"f{fr}", fill=(255, 230, 0))
            imgs.append(im)
    if not imgs:
        im = PILImage.new("RGB", (THUMB_H, THUMB_H), (40, 40, 40))
        imgs = [im]
    total_w = sum(i.width for i in imgs) + 4 * (len(imgs) - 1)
    canvas = PILImage.new("RGB", (total_w, THUMB_H), (25, 25, 25))
    x = 0
    for i in imgs:
        canvas.paste(i, (x, 0)); x += i.width + 4
    buf = BytesIO(); canvas.save(buf, format="JPEG", quality=85)
    return buf.getvalue()

print(f"Extracted thumbnails for {len(crops)} track IDs "
      f"(frames read: {len(need)}).")

In [ ]:
# ── Re-ID gallery : assign canonical jersey ID + team/role per raw track ────
_TEAM_SWATCH = {0: "#f472b6", 1: "#22d3ee", -1: "#9ca3af"}
_CLS_NAME    = {GK_CLASS: "GK", PLAYER_CLASS: "PL", REF_CLASS: "REF"}
_ROLE_OPTS   = [("Home (team 0)", "home"), ("Away (team 1)", "away"),
                ("Goalkeeper", "gk"), ("Referee", "ref"), ("Drop", "drop")]

def _default_role(row):
    if row.cls == REF_CLASS: return "ref"
    if row.cls == GK_CLASS:  return "gk"
    return "home" if row.team == 0 else ("away" if row.team == 1 else "home")

row_widgets = {}   # tid -> dict(canon, role)
gallery_rows = []

for _, r in summary.iterrows():
    tid = int(r.display_track_id)
    img = widgets.Image(value=strip_jpeg(tid), format="jpeg",
                        layout=widgets.Layout(height=f"{THUMB_H}px"))
    sw = _TEAM_SWATCH.get(int(r.team), "#9ca3af")
    meta = widgets.HTML(
        f"<div style='font-family:monospace;font-size:11px;line-height:1.5'>"
        f"<b>raw ID {tid}</b> &nbsp; <span style='color:{sw}'>&#9632;</span> "
        f"{_CLS_NAME.get(int(r.cls),'?')}<br>"
        f"frames <b>{int(r.f0)}–{int(r.f1)}</b> ({int(r.n)} det, {r.dur_s:.1f}s)<br>"
        f"pos (≈{r.cx:.0f}, {r.cy:.0f})px</div>")
    canon = widgets.IntText(value=tid, description="ID#",
                            layout=widgets.Layout(width="120px"))
    role  = widgets.Dropdown(options=_ROLE_OPTS, value=_default_role(r),
                             layout=widgets.Layout(width="150px"))
    row_widgets[tid] = {"canon": canon, "role": role, "cls": int(r.cls)}
    gallery_rows.append(widgets.HBox(
        [img, meta, canon, role],
        layout=widgets.Layout(border="1px solid #333", padding="3px",
                              align_items="center")))

w_apply  = widgets.Button(description="Apply mapping", button_style="primary",
                          layout=widgets.Layout(width="160px"))
w_report = widgets.Output()

gallery = widgets.VBox(
    [widgets.HTML("<b>Group fragments of the same player under one ID#. "
                  "Same player &rarr; same ID#. ID# = 0 drops the track.</b>")]
    + gallery_rows + [w_apply, w_report])
ipy_display(gallery)

In [ ]:
# ── Apply the mapping : rewrite display_track_id, team_id, collisions ──────
clean = {"df": None, "idmap": None}

_ROLE_TEAM = {"home": 0, "away": 1, "gk": -1, "ref": -1, "drop": -1}
_ROLE_CLS  = {"gk": GK_CLASS, "ref": REF_CLASS}   # role can override class

def _apply(_):
    with w_report:
        w_report.clear_output(wait=True)
        idmap = {}
        for tid, w in row_widgets.items():
            canon = int(w["canon"].value)
            role  = w["role"].value
            idmap[tid] = {"canon": canon, "role": role,
                          "team": _ROLE_TEAM[role]}

        d = df.copy()
        ppl = d["class_id"].isin(PEOPLE)
        d["new_id"]   = d["display_track_id"]
        d["new_team"] = d["team_id"]
        d["new_cls"]  = d["class_id"]
        drop_idx = []
        for tid, m in idmap.items():
            sel = ppl & (d["display_track_id"] == tid)
            if m["canon"] <= 0 or m["role"] == "drop":
                drop_idx.append(d.index[sel]); continue
            d.loc[sel, "new_id"]   = m["canon"]
            d.loc[sel, "new_team"] = m["team"]
            if m["role"] in _ROLE_CLS:
                d.loc[sel, "new_cls"] = _ROLE_CLS[m["role"]]
        if drop_idx:
            d = d.drop(index=np.concatenate([i.values for i in drop_idx]))

        # Collision: same canonical ID appears >1 time in a frame.
        ppl2 = d["class_id"].isin(PEOPLE) | d["new_cls"].isin(PEOPLE)
        coll = (d[ppl2].groupby(["frame", "new_id"]).size()
                .reset_index(name="k").query("k > 1"))
        n_coll = len(coll)

        # Resolve collisions: keep the highest-confidence box per (frame,new_id).
        if n_coll:
            d["_isppl"] = d["new_cls"].isin(PEOPLE)
            keep = (d[d["_isppl"]]
                    .sort_values("conf", ascending=False)
                    .drop_duplicates(["frame", "new_id"], keep="first").index)
            d = pd.concat([d[~d["_isppl"]], d.loc[keep]]).sort_index()
            d = d.drop(columns="_isppl")

        d["display_track_id"] = d["new_id"]
        d["team_id"]          = d["new_team"]
        d["class_id"]         = d["new_cls"]
        d = d.drop(columns=["new_id", "new_team", "new_cls"])
        clean["df"] = d.reset_index(drop=True)
        clean["idmap"] = idmap

        n_canon = len({m["canon"] for m in idmap.values()
                       if m["canon"] > 0 and m["role"] != "drop"})
        n_drop  = sum(1 for m in idmap.values()
                      if m["canon"] <= 0 or m["role"] == "drop")
        print(f"Raw track IDs        : {len(idmap)}")
        print(f"Canonical players    : {n_canon}")
        print(f"Dropped tracks       : {n_drop}")
        print(f"Frame collisions     : {n_coll} (kept highest-confidence box)")
        if n_coll:
            print("\nSample collision frames:")
            ipy_display(coll.head(10))
        print("\nMapping applied → run the next cell to interpolate gaps.")

w_apply.on_click(_apply)
print("Ready — press 'Apply mapping' in the gallery above.")

In [ ]:
# ── Interpolate short detection gaps per canonical ID ──────────────────────
def interpolate_gaps(d, max_gap=MAX_GAP):
    d = d.copy()
    if "track_filled" not in d.columns:
        d["track_filled"] = 0
    new_rows = []
    cols_lin = ["x1", "y1", "x2", "y2", "x_m", "y_m", "cx", "cy"]
    ppl = d[d["class_id"].isin(PEOPLE)]
    for cid, g in ppl.groupby("display_track_id"):
        g = g.sort_values("frame")
        frames = g["frame"].to_numpy()
        for a, b in zip(frames[:-1], frames[1:]):
            gap = int(b - a)
            if 1 < gap <= max_gap:
                ra = g[g["frame"] == a].iloc[0]
                rb = g[g["frame"] == b].iloc[0]
                for k in range(1, gap):
                    t = k / gap
                    row = ra.copy()
                    row["frame"] = a + k
                    for c in cols_lin:
                        if c in ra and pd.notna(ra[c]) and pd.notna(rb[c]):
                            row[c] = ra[c] + t * (rb[c] - ra[c])
                    row["conf"] = 0.0
                    row["detector_ran"] = 0
                    row["track_filled"] = 1
                    new_rows.append(row)
    if new_rows:
        d = pd.concat([d, pd.DataFrame(new_rows)], ignore_index=True)
    return d.sort_values(["frame", "class_id", "display_track_id"]).reset_index(drop=True)

if clean["df"] is None:
    print("Run 'Apply mapping' first.")
else:
    before = len(clean["df"])
    clean["df"] = interpolate_gaps(clean["df"])
    filled = int(clean["df"]["track_filled"].sum())
    print(f"Interpolated rows added : {filled}")
    print(f"Total rows {before} → {len(clean['df'])}")

In [ ]:
# ── Save cleaned tracks + the ID map ───────────────────────────────────────
if clean["df"] is None:
    print("Nothing to save — run the mapping + interpolation cells first.")
else:
    os.makedirs(os.path.dirname(CLEAN_CSV), exist_ok=True)
    clean["df"].to_csv(CLEAN_CSV, index=False)
    pd.DataFrame([
        {"raw_track_id": t, "canonical_id": m["canon"],
         "role": m["role"], "team_id": m["team"]}
        for t, m in clean["idmap"].items()
    ]).to_csv(IDMAP_CSV, index=False)

    d = clean["df"]
    ppl = d[d["class_id"].isin(PEOPLE)]
    print(f"Saved cleaned tracks → {CLEAN_CSV}  ({len(d)} rows)")
    print(f"Saved ID map         → {IDMAP_CSV}")
    print(f"\nCanonical players: {ppl['display_track_id'].nunique()}")
    print("Detections per team_id:")
    ipy_display(ppl.groupby("team_id").size().rename("rows"))
    print("Frames covered per player (top 10):")
    ipy_display(ppl.groupby("display_track_id")["frame"].nunique()
                .sort_values(ascending=False).head(10))